# External validation of the LC25000 CNN + BiMamba model (seed 42) on LungHist700 and Chaoyang

**External validation** = the model is trained **only on LC25000** and then tested on two other datasets
**without any training on them**. This shows whether the model works on images from other hospitals,
scanners and staining.

| External dataset | Classes that match LC25000 | Used images |
|---|---|---|
| **LungHist700** (20×) | adenocarcinoma → `lung_aca`, squamous cell carcinoma → `lung_scc`, normal → `lung_n` | all 3 classes (359 images, 38 patients) |
| **Chaoyang** (colon) | adenocarcinoma → `colon_aca`, normal → `colon_n` | only these 2 classes (serrated and adenoma do not exist in LC25000) |

**What the notebook does**

1. Gets the **seed-42 CNN + BiMamba** LC25000 model. It first looks for the saved file
   `weights_cnn_mamba_refgroup.pt` among the inputs. **If it is not found, it trains the model again**
   with exactly the same code, split and seed as the main run (about 2.5 h).
2. Trains a **ResNet-50 (ImageNet)** on the same LC25000 training split as a baseline (about 25 min),
   so the external result can be compared fairly.
3. Tests both models on LungHist700 and Chaoyang, **with and without stain normalisation**
   (Reinhard colour matching to LC25000). No external image is ever used for training.
   * *restricted* prediction: the model chooses only among the classes that exist in that dataset
     (3 lung classes, or 2 colon classes) — the main number;
   * *open* prediction: the model may choose any of its 5 LC25000 classes — a stricter check.
4. Runs the **XAI comparison** (influence_cc vs Grad-CAM, LIME, MambaLRP: stability, sanity, insertion,
   deletion, paired Wilcoxon + Holm) on external images.

**Inputs to attach:** LC25000, LungHist700 (Kaggle dataset `abdullahhasansajjad/lunghist700`, or the
notebook downloads it from figshare), Chaoyang (your private dataset), and, if you still have it, the
output of the main seed-42 notebook (contains `weights_cnn_mamba_refgroup.pt`). Internet **ON**, GPU T4.

**Time:** about 1 h with the saved weights, about 3.5 h without them.

In [1]:
QUICK_TEST = False          # True = a few minutes on tiny settings. Run this FIRST.

# ---------------- LC25000 model (same as the main run) ----------------
MODEL_NAME, SPLIT_NAME = "cnn_mamba", "refgroup"
EPOCHS        = 1 if QUICK_TEST else 12
BATCH_SIZE    = 8 if QUICK_TEST else 16
LR            = 3e-4
WEIGHT_DECAY  = 0.05
IMG_SIZE      = 224
NUM_CLASSES   = 5
SEED          = 42
MAX_PER_CLASS = 60 if QUICK_TEST else 5000
D_MODEL, DEPTH, D_STATE, D_CONV, EXPAND, PATCH = 192, 4, 8, 4, 2, 2
CHUNK_L        = 32
USE_CHECKPOINT = True
REF_URL = ("https://raw.githubusercontent.com/GeorgeBatch/LC25000-clean/main/"
           "kaggle/lc25000_image_groups.csv")
SPLIT_ON = "reference"
USE_SYNTHETIC_IF_MISSING = False

# ---------------- ResNet-50 baseline ----------------
TRAIN_RESNET   = True
RESNET_EPOCHS  = 1 if QUICK_TEST else 5
RESNET_LR      = 1e-4

# ---------------- external data ----------------
LH_RESOLUTION = "20x"
FIGSHARE_URL  = "https://ndownloader.figshare.com/files/45206104"     # LungHist700.rar (670 MB)
EXT_MAX_PER_CLASS = 10 if QUICK_TEST else None                        # None = all images

# ---------------- XAI on external images ----------------
XAI_PER_CLASS  = 2 if QUICK_TEST else 20         # per class and per dataset (LungHist 3x20, Chaoyang 2x20)
LIME_SAMPLES   = 40 if QUICK_TEST else 150
DELETION_STEPS = 10 if QUICK_TEST else 25
TOP_FRAC       = 0.20
N_BOOTSTRAP    = 200 if QUICK_TEST else 10000
REFERENCE      = "influence_cc"
ALPHA          = 0.05
PER_CLASS, N_XAI_IMAGES = XAI_PER_CLASS, XAI_PER_CLASS * 5

OUT_DIR = "/kaggle/working/results_external"
print(f"QUICK_TEST={QUICK_TEST}  seed={SEED}  resnet={TRAIN_RESNET}")

QUICK_TEST=False  seed=42  resnet=True


## Section 2 — Imports, seeds, GPU

Every reachable seed is fixed. This matters more than usual here: one of the claims concerns
the *variability* of explanations, which cannot be measured on top of uncontrolled noise.

In [2]:
import os, math, random, time, warnings, glob, copy
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
from collections import defaultdict

import numpy as np, pandas as pd, torch
import torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

def set_seed(s=SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
set_seed()

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
os.makedirs(OUT_DIR, exist_ok=True)
def free_gpu():
    if DEVICE == "cuda": torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
def gpu_peak_gb():
    return torch.cuda.max_memory_allocated()/1024**3 if DEVICE == "cuda" else 0.0

print("torch:", torch.__version__, "| device:", DEVICE)
if DEVICE == "cuda": print("gpu  :", torch.cuda.get_device_name(0))
print("output:", OUT_DIR)

torch: 2.11.0+cu128 | device: cuda
gpu  : Tesla T4
output: /kaggle/working/results_external


## Section 3 — Dataset

LC25000's five classes. We search all of `/kaggle/input` for the folder names so the notebook
survives a change in the dataset's directory layout.

`lung_aca` versus `lung_scc` is the pair that every recent paper in this area still confuses.
It gets its own analysis in Section 7.

In [3]:
CLASSES = ["lung_aca","lung_scc","lung_n","colon_aca","colon_n"]
CLASS_TO_IDX = {c:i for i,c in enumerate(CLASSES)}

def find_class_dirs(root="/kaggle/input"):
    found = {}
    if not os.path.isdir(root): return found
    for dp, dns, _ in os.walk(root):
        for d in dns:
            if d in CLASSES and d not in found: found[d] = os.path.join(dp, d)
    return found

class_dirs = find_class_dirs()
SYNTHETIC = len(class_dirs) < len(CLASSES)
SYN_DIR = "/kaggle/working/_syn"

if SYNTHETIC:
    if not USE_SYNTHETIC_IF_MISSING:
        raise FileNotFoundError("LC25000 not found. Attach it via 'Add Input'.")
    print("!! LC25000 NOT FOUND -> synthetic images.")
    print("!! The pipeline will run, but every number is meaningless.")
    rng = np.random.default_rng(SEED); rows = []
    npar = max(2, max(20, MAX_PER_CLASS)//20)
    for ci, c in enumerate(CLASSES):
        d = os.path.join(SYN_DIR, c); os.makedirs(d, exist_ok=True)
        for p in range(npar):
            base = rng.integers(0,255,(96,96,3),dtype=np.uint8)
            base = (base*0.4 + np.array([200-30*ci,150,190])*0.6).astype(np.uint8)
            for k in range(20):
                img = np.rot90(base, k%4)
                if k >= 10: img = img[:, ::-1]
                fp = os.path.join(d, f"p{p:04d}_c{k:02d}.png")
                if not os.path.exists(fp):
                    Image.fromarray(np.ascontiguousarray(img)).resize((128,128)).save(fp)
                rows.append({"path":fp,"cls":c,"y":ci})
    df = pd.DataFrame(rows)
else:
    print("LC25000 found:")
    rows = []
    for c in CLASSES:
        fs = sorted(glob.glob(os.path.join(class_dirs[c],"*.jpeg")) +
                    glob.glob(os.path.join(class_dirs[c],"*.jpg")))[:MAX_PER_CLASS]
        print(f"  {c:10s} {len(fs):5d}")
        rows += [{"path":f,"cls":c,"y":CLASS_TO_IDX[c]} for f in fs]
    df = pd.DataFrame(rows)

df = df.reset_index(drop=True)
print("total images:", len(df))

LC25000 found:
  lung_aca    5000
  lung_scc    5000
  lung_n      5000
  colon_aca   5000
  colon_n     5000
total images: 25000


In [4]:
# KMeans audit is not needed here (it is in the main notebook)
from sklearn.model_selection import GroupShuffleSplit
import urllib.request
df["group_kmeans"] = df["cls"] + "_na"

In [5]:
# ---------- step 2a: load the published parent labels ----------
def load_reference():
    hits = glob.glob("/kaggle/input/**/lc25000_image_groups.csv", recursive=True)
    if hits:
        print("reference labels found in inputs:", hits[0]); return pd.read_csv(hits[0])
    dst = f"{OUT_DIR}/lc25000_image_groups.csv"
    try:
        urllib.request.urlretrieve(REF_URL, dst)
        print("reference labels downloaded from GitHub"); return pd.read_csv(dst)
    except Exception as e:
        print("download failed:", repr(e)); return None

ref = None if SYNTHETIC else load_reference()

if ref is None:
    if not SYNTHETIC:
        raise RuntimeError(
            "Could not get the LC25000-clean labels. Turn Internet ON in the notebook settings "
            "(or attach lc25000_image_groups.csv as a dataset) and run again. We stop here on "
            "purpose: the KMeans-only split leaks and must not be the headline result.")
    print("!! synthetic data: no reference labels exist; using KMeans groups as a stand-in.")
    df["group_ref"] = df["group_kmeans"]
else:
    need = {"filename", "label", "group_id"}
    assert need <= set(ref.columns), f"reference CSV is missing {need - set(ref.columns)}"
    assert ref.filename.is_unique
    assert ref.groupby("group_id").label.nunique().max() == 1, "a reference parent spans two classes"
    df["filename"] = df.path.map(os.path.basename)
    assert df.filename.is_unique, "two images share a file name"
    j = df.merge(ref[["filename", "label", "group_id"]], on="filename", how="left",
                 validate="one_to_one")
    assert len(j) == len(df)
    missing = int(j.group_id.isna().sum())
    assert missing == 0, f"{missing} images have no reference label (file names differ?)"
    bad = int((j.label != j.cls).sum())
    assert bad == 0, f"{bad} images have a different class in the reference file"
    df["group_ref"] = "ref_" + j["group_id"].astype(int).astype(str).to_numpy()
    print(f"joined {len(df)}/{len(df)} images, every class label agrees")

print("KMeans groups:", df.group_kmeans.nunique(), "| reference groups:", df.group_ref.nunique())
_sz = df.groupby("group_ref").size()
print(f"reference group size: mean {_sz.mean():.1f}, min {_sz.min()}, max {_sz.max()} "
      + ("(about 20 expected on the full dataset)" if not QUICK_TEST else "(QUICK_TEST subset: small groups are normal)"))

reference labels downloaded from GitHub
joined 25000/25000 images, every class label agrees
KMeans groups: 5 | reference groups: 1246
reference group size: mean 20.1, min 1, max 53 (about 20 expected on the full dataset)


In [6]:
# ---------- the same leakage-free split as the main run (LC25000-clean groups, seed 42) ----------
def group_split(frame, col):
    g1 = GroupShuffleSplit(1, test_size=0.30, random_state=SEED)
    a, b = next(g1.split(frame, frame.y, groups=frame[col]))
    trn, tmp_ = frame.iloc[a], frame.iloc[b]
    g2 = GroupShuffleSplit(1, test_size=0.50, random_state=SEED)
    v, t = next(g2.split(tmp_, tmp_.y, groups=tmp_[col]))
    return trn, tmp_.iloc[v], tmp_.iloc[t]
tr, va, te = group_split(df, "group_ref")
for a_, b_, nm in [(tr, te, "train/test"), (tr, va, "train/val"), (va, te, "val/test")]:
    assert not set(a_.group_ref) & set(b_.group_ref), f"{nm} share a parent"
print(f"LC25000 split sizes {len(tr)}/{len(va)}/{len(te)}   (main run: 17399/3778/3823)")

LC25000 split sizes 17399/3778/3823   (main run: 17399/3778/3823)


In [7]:
import torchvision.transforms as T
MEAN, STD = [0.485,0.456,0.406], [0.229,0.224,0.225]

# augmentation is kept mild: heavy augmentation would partly re-create the duplicate
# problem we just removed
train_tf = T.Compose([T.Resize((IMG_SIZE,IMG_SIZE)), T.RandomHorizontalFlip(),
                      T.RandomVerticalFlip(), T.ColorJitter(0.1,0.1,0.1),
                      T.ToTensor(), T.Normalize(MEAN,STD)])
eval_tf  = T.Compose([T.Resize((IMG_SIZE,IMG_SIZE)), T.ToTensor(), T.Normalize(MEAN,STD)])

class HistoDataset(Dataset):
    def __init__(self, frame, tf):
        self.paths, self.labels, self.tf = frame.path.tolist(), frame.y.tolist(), tf
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        return self.tf(Image.open(self.paths[i]).convert("RGB")), self.labels[i]

nw = 2 if QUICK_TEST else 4
tr_ld = DataLoader(HistoDataset(tr, train_tf), BATCH_SIZE, shuffle=True,
                   num_workers=nw, pin_memory=True, drop_last=True)
va_ld = DataLoader(HistoDataset(va, eval_tf), BATCH_SIZE, num_workers=nw, pin_memory=True)
te_ld = DataLoader(HistoDataset(te, eval_tf), BATCH_SIZE, num_workers=nw, pin_memory=True)
print("loaders ready")

loaders ready


## Section 5 — The selective scan

A Mamba layer is the recurrence

$$h_t = \bar{A}_t h_{t-1} + \bar{B}_t u_t, \qquad y_t = C_t h_t + D u_t$$

As a Python loop over 196 tokens this is far too slow to train, so we evaluate it with a
**parallel scan** — the recurrence is associative, so it needs $\log_2$ steps instead of $L$.
That is why this notebook does not need the `mamba-ssm` CUDA package, which is painful to
compile on Kaggle.

**Chunking is what prevents CUDA out-of-memory.** A naive full-length parallel scan builds
`(batch, d_inner, length, d_state)` tensors once per scan step and autograd keeps every one of
them — roughly 7.5 GB across 8 blocks, which kills a 16 GB GPU. We cut the sequence into
`CHUNK_L` blocks: parallel inside a chunk, sequential across chunks, carrying the state with
`exp(cumsum(delta*A))`, which is exact. Combined with gradient checkpointing this drops peak
training memory by about 12x, to under 2 GB.

The cell below also keeps a plain sequential implementation and **asserts the two agree at
every chunk size**. A silent bug here would invalidate the entire thesis, so it is worth the
three seconds.

In [8]:
def _shift_right(x, n, fill):
    if n >= x.shape[2]: return torch.full_like(x, fill)
    return torch.cat([torch.full_like(x[:,:,:n], fill), x[:,:,:-n]], dim=2)

def _inclusive_scan(a, x):
    """T_j = a_j * T_{j-1} + x_j, in log2(L) vectorised steps."""
    l, off = a.shape[2], 1
    while off < l:
        x = x + a*_shift_right(x, off, 0.0)
        a = a*_shift_right(a, off, 1.0)
        off *= 2
    return x

def selective_scan_parallel(u, delta, A, B, C, D, chunk=None):
    """u,delta:(b,d,l)  A:(d,n)  B,C:(b,n,l)  D:(d,) -> y:(b,d,l)
    Parallel inside a chunk, sequential across chunks. Mathematically identical
    to the full scan; the carry exp(cumsum(delta*A)) is exact."""
    chunk = chunk or CHUNK_L
    b, d, l = u.shape
    log_a = delta.unsqueeze(-1)*A[:,None,:]
    x_in  = delta.unsqueeze(-1)*B.transpose(1,2).unsqueeze(1)*u.unsqueeze(-1)
    h_prev, ys = u.new_zeros(b,d,A.shape[-1]), []
    for s0 in range(0, l, chunk):
        s1 = min(s0+chunk, l); la = log_a[:,:,s0:s1]
        h = _inclusive_scan(la.exp(), x_in[:,:,s0:s1]) + la.cumsum(2).exp()*h_prev.unsqueeze(2)
        ys.append(torch.einsum("bdln,bnl->bdl", h, C[:,:,s0:s1]))
        h_prev = h[:,:,-1]
    return torch.cat(ys,2) + u*D[None,:,None]

def selective_scan_sequential(u, delta, A, B, C, D):
    """Slow reference. Only used to verify the fast path."""
    b, d, l = u.shape
    a = torch.exp(delta.unsqueeze(-1)*A[:,None,:])
    x = delta.unsqueeze(-1)*B.transpose(1,2).unsqueeze(1)*u.unsqueeze(-1)
    h = u.new_zeros(b, d, A.shape[-1]); ys = []
    for t in range(l):
        h = a[:,:,t]*h + x[:,:,t]
        ys.append(torch.einsum("bdn,bn->bd", h, C[:,:,t]))
    return torch.stack(ys,-1) + u*D[None,:,None]

# ---- verification --------------------------------------------------------
torch.manual_seed(0)
_u  = torch.randn(2,6,64); _dl = F.softplus(torch.randn(2,6,64))
_A  = -torch.rand(6,4)-0.1; _B = torch.randn(2,4,64)
_C  = torch.randn(2,4,64);  _D = torch.randn(6)
_ref = selective_scan_sequential(_u,_dl,_A,_B,_C,_D)
for _ck in [1,4,7,32,1000]:
    _e = (selective_scan_parallel(_u,_dl,_A,_B,_C,_D,chunk=_ck)-_ref).abs().max().item()
    print(f"  chunk={_ck:<5d} max |chunked - sequential| = {_e:.2e}")
    assert _e < 1e-4, f"Chunked scan wrong at chunk={_ck} - do not trust any result."
print("OK: chunked scan verified against the sequential reference.")

  chunk=1     max |chunked - sequential| = 0.00e+00
  chunk=4     max |chunked - sequential| = 1.91e-06
  chunk=7     max |chunked - sequential| = 1.91e-06
  chunk=32    max |chunked - sequential| = 1.91e-06
  chunk=1000  max |chunked - sequential| = 2.38e-06
OK: chunked scan verified against the sequential reference.


## Section 6 — The model

A deliberately plain bidirectional CNN+Mamba. No attention modules, no handcrafted texture
features, no ensembling — it is the **instrument**, not the contribution. Adding an attention
block would also wreck the argument, since a reviewer would ask whether the explanation came
from the SSM or from the attention.

The CNN stem is kept small on purpose: its receptive field should span a few nuclei, not a
whole gland. Long-range aggregation is the Mamba stack's job.

**`capture=True` returns `C`, the gate and `out_proj` as well as `delta`, `A`, `B`, `u`.** Those
first three are exactly what Run 1's score threw away, and exactly what carries class
information to the output. Note in `forward` that `delta`, `B` and `C` are all computed *from
the input* — which is why our score is an approximation and why Section 10 measures the gap
rather than assuming it away.

A tissue patch has no natural reading order, so each layer runs one forward scan and one
reverse scan. With a single direction the bottom-right of the image would receive top-left
context but not the reverse, which is an artefact of the scan rather than of the biology.

In [9]:
def _silu_lrp(x):
    """Eq. 4: SiLU with the sigmoid detached, restoring relevance conservation."""
    return x * torch.sigmoid(x).detach()


class MambaBlock(nn.Module):
    def __init__(self, d_model, d_state=D_STATE, d_conv=D_CONV, expand=EXPAND):
        super().__init__()
        d_inner = expand*d_model
        self.d_inner, self.d_state, self.d_conv = d_inner, d_state, d_conv
        self.in_proj = nn.Linear(d_model, d_inner*2, bias=False)
        self.conv1d  = nn.Conv1d(d_inner, d_inner, d_conv, groups=d_inner, padding=d_conv-1)
        self.dt_rank = math.ceil(d_model/16)
        self.x_proj  = nn.Linear(d_inner, self.dt_rank+2*d_state, bias=False)
        self.dt_proj = nn.Linear(self.dt_rank, d_inner, bias=True)
        dt = torch.exp(torch.rand(d_inner)*(math.log(0.1)-math.log(1e-3))+math.log(1e-3)).clamp(min=1e-4)
        with torch.no_grad(): self.dt_proj.bias.copy_(dt + torch.log(-torch.expm1(-dt)))
        A = torch.arange(1, d_state+1, dtype=torch.float32).repeat(d_inner,1)
        self.A_log = nn.Parameter(torch.log(A))      # A kept negative -> stable decay
        self.D = nn.Parameter(torch.ones(d_inner))
        self.out_proj = nn.Linear(d_inner, d_model, bias=False)

    def forward(self, x, capture=False):
        b, l, _ = x.shape
        u, z = self.in_proj(x).chunk(2,-1)
        u = F.silu(self.conv1d(u.transpose(1,2))[..., :l])
        x_dbl = self.x_proj(u.transpose(1,2))                    # delta, B, C come FROM u
        delta, B, C = torch.split(x_dbl, [self.dt_rank, self.d_state, self.d_state], -1)
        delta = F.softplus(self.dt_proj(delta)).transpose(1,2)
        B, C = B.transpose(1,2).contiguous(), C.transpose(1,2).contiguous()
        A = -torch.exp(self.A_log.float())
        y = selective_scan_parallel(u, delta, A, B, C, self.D.float())
        gate = F.silu(z.transpose(1,2))
        out = self.out_proj((y*gate).transpose(1,2))
        if capture:
            return out, {"delta":delta.detach(), "A":A.detach(), "B":B.detach(),
                         "C":C.detach(), "u":u.detach(), "gate":gate.detach(),
                         "D":self.D.detach(), "W_out":self.out_proj.weight.detach()}
        return out, None

    def forward_lrp(self, x):
        """MambaLRP forward pass (Jafari et al., NeurIPS 2024).

        Three modifications, exactly as the paper specifies:
          Eq. 4     SiLU        y = x * [sigma(x)]_cst
          Eq. 6-7   SSM         Abar_t, Bbar_t, C_t all treated as constants
          Eq. 9     gate        y = 0.5*(zA*zB) + 0.5*[zA*zB]_cst
        Relevance is then read off as gradient x input. LayerNorm is left standard,
        since the paper's stated rules do not cover it.
        """
        b, l, _ = x.shape
        u, z = self.in_proj(x).chunk(2, -1)
        u = _silu_lrp(self.conv1d(u.transpose(1, 2))[..., :l])
        x_dbl = self.x_proj(u.transpose(1, 2))
        delta, B, C = torch.split(x_dbl, [self.dt_rank, self.d_state, self.d_state], -1)
        # ---- Eq. 6-7: the input-dependent SSM parameters become constants ----
        delta = F.softplus(self.dt_proj(delta)).transpose(1, 2).detach()
        B = B.transpose(1, 2).contiguous().detach()
        C = C.transpose(1, 2).contiguous().detach()
        A = (-torch.exp(self.A_log.float())).detach()
        y = selective_scan_parallel(u, delta, A, B, C, self.D.float())
        # ---- Eq. 9: half-relevance through the multiplicative gate ----
        prod = y * _silu_lrp(z.transpose(1, 2))
        return self.out_proj((0.5*prod + 0.5*prod.detach()).transpose(1, 2))


class BiMambaLayer(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.norm = nn.LayerNorm(d_model)
        self.fwd, self.bwd = MambaBlock(d_model), MambaBlock(d_model)
    def forward(self, x, capture=False):
        h = self.norm(x)
        yf, cf = self.fwd(h, capture); yb, cb = self.bwd(h.flip(1), capture)
        return x + yf + yb.flip(1), ({"fwd":cf,"bwd":cb} if capture else None)
    def forward_lrp(self, x):
        h = self.norm(x)
        return x + self.fwd.forward_lrp(h) + self.bwd.forward_lrp(h.flip(1)).flip(1)


def _ckpt_layer(layer, t): return layer(t, False)[0]

class CNNStem(nn.Module):
    """224 -> 28 via three stride-2 blocks. Local morphology only."""
    def __init__(self, dims=(64,128,128)):
        super().__init__()
        ch, L = (3,)+dims, []
        for i in range(len(dims)):
            L += [nn.Conv2d(ch[i],ch[i+1],3,2,1,bias=False), nn.BatchNorm2d(ch[i+1]), nn.GELU(),
                  nn.Conv2d(ch[i+1],ch[i+1],3,1,1,bias=False), nn.BatchNorm2d(ch[i+1]), nn.GELU()]
        self.net, self.out_dim = nn.Sequential(*L), dims[-1]
    def forward(self, x): return self.net(x)

class CNNMamba(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES, d_model=D_MODEL, depth=DEPTH):
        super().__init__()
        self.stem = CNNStem()
        self.patch_embed = nn.Conv2d(self.stem.out_dim, d_model, PATCH, PATCH)
        self.grid = IMG_SIZE//8//PATCH               # 224 -> 28 -> 14 (196 tokens, as ViT-B/16)
        self.num_tokens = self.grid**2
        self.layers = nn.ModuleList([BiMambaLayer(d_model) for _ in range(depth)])
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, num_classes)
        self.apply(self._init)
    @staticmethod
    def _init(m):
        if isinstance(m, nn.Linear):
            nn.init.trunc_normal_(m.weight, std=0.02)
            if m.bias is not None: nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Conv2d):
            nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
    def tokenize(self, x):
        return self.patch_embed(self.stem(x)).flatten(2).transpose(1,2)
    def from_tokens(self, t, capture=False):
        caps = []
        for layer in self.layers:
            if USE_CHECKPOINT and self.training and not capture and t.requires_grad:
                t = torch.utils.checkpoint.checkpoint(_ckpt_layer, layer, t, use_reentrant=False)
            else:
                t, c = layer(t, capture)
                if capture: caps.append(c)
        return self.head(self.norm(t).mean(1)), caps
    def forward(self, x, capture=False):
        lo, caps = self.from_tokens(self.tokenize(x), capture)
        return (lo, caps) if capture else lo
    def from_tokens_lrp(self, t):
        for layer in self.layers:
            t = layer.forward_lrp(t)
        return self.head(self.norm(t).mean(1))

print("parameters:", sum(p.numel() for p in CNNMamba().parameters())/1e6, "M")

parameters: 2.595781 M


## The LC25000 models

In [10]:
# ---------- get the seed-42 CNN + BiMamba: load saved weights, or train again ----------
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
@torch.no_grad()
def evaluate(m, loader):
    m.eval(); P, Yl = [], []
    for x, y in loader:
        o = m(x.to(DEVICE, non_blocking=True))
        if isinstance(o, tuple): o = o[0]
        P.append(o.argmax(1).cpu()); Yl.append(y)
    P, Yl = torch.cat(P).numpy(), torch.cat(Yl).numpy()
    return accuracy_score(Yl, P), f1_score(Yl, P, average="macro"), confusion_matrix(Yl, P, labels=list(range(NUM_CLASSES)))

def train_lc(make, epochs, lr, tag):
    set_seed(); free_gpu()
    m = make().to(DEVICE)
    opt = torch.optim.AdamW(m.parameters(), lr=lr, weight_decay=WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(1, epochs))
    crit = nn.CrossEntropyLoss(label_smoothing=0.05)
    best, best_state = -1.0, None
    for ep in range(epochs):
        m.train(); t0 = time.time()
        for x, y in tr_ld:
            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
            o = m(x); o = o[0] if isinstance(o, tuple) else o
            loss = crit(o, y); opt.zero_grad(set_to_none=True); loss.backward()
            nn.utils.clip_grad_norm_(m.parameters(), 1.0); opt.step()
        sched.step()
        vacc, _, _ = evaluate(m, va_ld)
        if vacc > best: best, best_state = vacc, {k: v.detach().cpu().clone() for k, v in m.state_dict().items()}
        print(f"  {tag} epoch {ep+1}/{epochs}  val_acc {vacc:.4f} ({time.time()-t0:.0f}s)", flush=True)
    m.load_state_dict(best_state); return m.eval()

_hits = sorted(glob.glob("/kaggle/input/**/weights_cnn_mamba_refgroup.pt", recursive=True))
if _hits and not QUICK_TEST:
    print("loading saved seed-42 weights:", _hits[0])
    model = CNNMamba().to(DEVICE); model.load_state_dict(torch.load(_hits[0], map_location=DEVICE)); model.eval()
    WEIGHT_SOURCE = _hits[0]
else:
    print("saved weights not found -> training the seed-42 CNN + BiMamba again (same code as the main run)")
    model = train_lc(CNNMamba, EPOCHS, LR, "cnn_mamba")
    torch.save(model.state_dict(), f"{OUT_DIR}/weights_cnn_mamba_refgroup.pt")
    WEIGHT_SOURCE = "trained in this notebook"
acc_lc, f1_lc, cm_lc = evaluate(model, te_ld)
print(f"CNN + BiMamba on the LC25000 test set: acc {acc_lc:.4f}  macro-F1 {f1_lc:.4f}   (main run: 0.9806 / 0.9790)")

saved weights not found -> training the seed-42 CNN + BiMamba again (same code as the main run)
  cnn_mamba epoch 1/12  val_acc 0.9068 (740s)
  cnn_mamba epoch 2/12  val_acc 0.9288 (740s)
  cnn_mamba epoch 3/12  val_acc 0.9539 (740s)
  cnn_mamba epoch 4/12  val_acc 0.9561 (741s)
  cnn_mamba epoch 5/12  val_acc 0.9246 (741s)
  cnn_mamba epoch 6/12  val_acc 0.9653 (741s)
  cnn_mamba epoch 7/12  val_acc 0.9637 (741s)
  cnn_mamba epoch 8/12  val_acc 0.9669 (741s)
  cnn_mamba epoch 9/12  val_acc 0.9696 (741s)
  cnn_mamba epoch 10/12  val_acc 0.9709 (741s)
  cnn_mamba epoch 11/12  val_acc 0.9661 (741s)
  cnn_mamba epoch 12/12  val_acc 0.9685 (741s)
CNN + BiMamba on the LC25000 test set: acc 0.9882  macro-F1 0.9871   (main run: 0.9806 / 0.9790)


In [11]:
# ---------- ResNet-50 (ImageNet) baseline on the same LC25000 split ----------
import torchvision
def make_resnet():
    r = torchvision.models.resnet50(weights=torchvision.models.ResNet50_Weights.IMAGENET1K_V2)
    r.fc = nn.Linear(r.fc.in_features, NUM_CLASSES); return r
MODELS_EXT = {"CNN + BiMamba (ours)": model}
if TRAIN_RESNET:
    _rh = sorted(glob.glob("/kaggle/input/**/weights_resnet50_lc25000_seed42.pt", recursive=True))
    if _rh:
        resnet = make_resnet().to(DEVICE); resnet.load_state_dict(torch.load(_rh[0], map_location=DEVICE)); resnet.eval()
        print("loaded saved ResNet-50:", _rh[0])
    else:
        resnet = train_lc(make_resnet, RESNET_EPOCHS, RESNET_LR, "resnet50")
        torch.save(resnet.state_dict(), f"{OUT_DIR}/weights_resnet50_lc25000_seed42.pt")
    a_, f_, _ = evaluate(resnet, te_ld)
    print(f"ResNet-50 on the LC25000 test set: acc {a_:.4f}  macro-F1 {f_:.4f}")
    MODELS_EXT["ResNet-50 (ImageNet)"] = resnet
pd.DataFrame([dict(model=k, lc25000_test_acc=round(evaluate(v, te_ld)[0], 4)) for k, v in MODELS_EXT.items()]
             ).to_csv(f"{OUT_DIR}/table0_lc25000_test.csv", index=False)

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 187MB/s]


  resnet50 epoch 1/5  val_acc 0.9677 (215s)
  resnet50 epoch 2/5  val_acc 0.9690 (214s)
  resnet50 epoch 3/5  val_acc 0.9717 (214s)
  resnet50 epoch 4/5  val_acc 0.9775 (212s)
  resnet50 epoch 5/5  val_acc 0.9794 (214s)
ResNet-50 on the LC25000 test set: acc 0.9861  macro-F1 0.9851


## External datasets

In [12]:
# ---------- load the two external datasets ----------
import json, re, subprocess, shutil
LC = {c: i for i, c in enumerate(CLASSES)}       # lung_aca 0, lung_scc 1, lung_n 2, colon_aca 3, colon_n 4

# LungHist700 ----------------------------------------------------------
def _find_lh_csv(root="/kaggle/input"):
    for p in glob.glob(f"{root}/**/data.csv", recursive=True):
        try: cols = set(pd.read_csv(p, nrows=2).columns)
        except Exception: continue
        if {"superclass", "resolution", "image_id", "patient_id"} <= cols: return p
    return None
def _download_lh(dst="/kaggle/working/lunghist700_raw"):
    os.makedirs(dst, exist_ok=True); rar = f"{dst}/LungHist700.rar"
    if not os.path.exists(rar):
        print("downloading LungHist700 from figshare (670 MB) ..."); urllib.request.urlretrieve(FIGSHARE_URL, rar)
    for tool, args in [("unrar", ["x", "-o+", rar, dst + "/"]), ("7z", ["x", "-y", f"-o{dst}", rar]), ("bsdtar", ["-xf", rar, "-C", dst])]:
        if shutil.which(tool): subprocess.run([tool] + args, check=True, stdout=subprocess.DEVNULL); return dst
    raise RuntimeError("cannot extract LungHist700.rar; attach the Kaggle dataset abdullahhasansajjad/lunghist700")
LH_CSV = _find_lh_csv() or _find_lh_csv(_download_lh())
assert LH_CSV, "LungHist700 data.csv not found"
jpg = {}
for r in {"/kaggle/input", os.path.dirname(os.path.dirname(LH_CSV)), "/kaggle/working/lunghist700_raw"}:
    for p in glob.glob(f"{r}/**/*.jp*g", recursive=True): jpg.setdefault(os.path.basename(p), p)
meta = pd.read_csv(LH_CSV)
meta["superclass"] = meta.superclass.astype(str).str.strip().str.lower()
has_sub = "subclass" in meta.columns
def _fname(r):
    css = r.superclass if (not has_sub or pd.isnull(r.subclass)) else f"{r.superclass}_{r.subclass}"
    return f"{css}_{r.resolution}_{r.image_id}.jpg"
meta["path"] = meta.apply(_fname, axis=1).map(jpg)
lh = meta[meta.path.notna() & (meta.resolution == LH_RESOLUTION)].copy()
LH_MAP = {"aca": "lung_aca", "scc": "lung_scc", "nor": "lung_n"}
lh["lc_class"] = lh.superclass.map(LH_MAP); lh = lh[lh.lc_class.notna()]
lh["y"] = lh.lc_class.map(LC); lh["group"] = "patient_" + lh.patient_id.astype(str); lh["dataset"] = "LungHist700"

# Chaoyang -------------------------------------------------------------
def _ff(name):
    h = sorted(glob.glob(f"/kaggle/input/**/{name}", recursive=True)); return h[0] if h else None
rows = []
_tj, _sj = _ff("train.json"), _ff("test.json")
CY_NAMES = {"normal": 0, "serrated": 1, "adenocarcinoma": 2, "adenoma": 3,
            "label_0": 0, "label_1": 1, "label_2": 2, "label_3": 3, "0": 0, "1": 1, "2": 2, "3": 3}
if _tj and _sj:                                   # layout A: official train.json / test.json
    CY_ROOT = os.path.dirname(_tj)
    _all = {}                                     # images are searched ONLY inside the Chaoyang folder
    for p in glob.glob(f"{CY_ROOT}/**/*", recursive=True):
        if p.lower().endswith((".jpg", ".jpeg", ".png")): _all.setdefault("/".join(p.replace("\\", "/").split("/")[-2:]).lower(), p)
    for jp in [_tj, _sj]:
        for e in json.load(open(jp)):
            rel = "/".join(e["name"].replace("\\", "/").split("/")[-2:]).lower()
            rows.append(dict(path=_all.get(rel), label=int(e["label"]), split=os.path.basename(jp)[:-5],
                             filename=os.path.basename(e["name"])))
else:                                             # layout B: chaoyang/{train,valid,test}/<class folder>/image
    _cands = [d for d in glob.glob("/kaggle/input/**/", recursive=True)
              if os.path.basename(os.path.normpath(d)).lower() == "chaoyang"]
    assert _cands, "Chaoyang not found: no train.json/test.json and no folder named 'chaoyang'"
    CY_ROOT = os.path.normpath(_cands[0])
    for p in glob.glob(f"{CY_ROOT}/**/*", recursive=True):
        if not p.lower().endswith((".jpg", ".jpeg", ".png")): continue
        parts = os.path.relpath(p, CY_ROOT).replace("\\", "/").split("/")
        lab = next((CY_NAMES[q.lower()] for q in reversed(parts[:-1]) if q.lower() in CY_NAMES), None)
        if lab is None:                          # last resort: the "-<label>.jpg" suffix in Chaoyang file names
            m_ = re.search(r"-(\d)\.[A-Za-z]+$", parts[-1]); lab = int(m_.group(1)) if m_ else None
        if lab is None: continue
        rows.append(dict(path=p, label=lab, split=parts[0], filename=parts[-1]))
print("Chaoyang folder:", CY_ROOT)
cy = pd.DataFrame(rows)
assert len(cy), "no Chaoyang images with a readable label were found"
print(f"Chaoyang label entries {len(cy)} | image files found {int(cy.path.notna().sum())}")
print("images per split and label:\n" + cy.pivot_table(index="label", columns="split", values="filename", aggfunc="size").fillna(0).astype(int).to_string())
assert cy.path.notna().mean() > 0.95, "most Chaoyang images were not found"
cy = cy[cy.path.notna()]
cy = cy.drop_duplicates("filename")              # an image that appears in two splits is tested once
CY_MAP = {0: "colon_n", 2: "colon_aca"}                 # 0 normal, 2 adenocarcinoma (1 serrated, 3 adenoma: not in LC25000)
cy["lc_class"] = cy.label.map(CY_MAP); cy = cy[cy.lc_class.notna()].copy()
cy["y"] = cy.lc_class.map(LC); cy["group"] = "case_" + cy.filename.str.split("-").str[0]; cy["dataset"] = "Chaoyang"

EXT = pd.concat([lh[["path", "lc_class", "y", "group", "dataset"]], cy[["path", "lc_class", "y", "group", "dataset"]]],
                ignore_index=True)
if EXT_MAX_PER_CLASS:
    EXT = EXT.groupby(["dataset", "lc_class"]).head(EXT_MAX_PER_CLASS).reset_index(drop=True)
EXT_CLASSES = {"LungHist700": [LC["lung_aca"], LC["lung_scc"], LC["lung_n"]], "Chaoyang": [LC["colon_aca"], LC["colon_n"]]}
summ = EXT.groupby(["dataset", "lc_class"]).agg(images=("path", "size"), groups=("group", "nunique"))
print(summ.to_string()); summ.to_csv(f"{OUT_DIR}/table1_external_data.csv")

Chaoyang folder: /kaggle/input/competitions/histopathology/archive/chaoyang
Chaoyang label entries 6160 | image files found 6160
images per split and label:
split  test  train  valid
label                    
0       705    999    112
1       321    757     85
2       840   1263    141
3       273    597     67
                       images  groups
dataset     lc_class                 
Chaoyang    colon_aca    2244      23
            colon_n      1816      66
LungHist700 lung_aca      146      16
            lung_n         85      11
            lung_scc      128      18


In [13]:
# ---------- stain normalisation (Reinhard: match the mean and std of LC25000 in LAB colour space) ----------
from skimage.color import rgb2lab, lab2rgb
_rs = np.random.RandomState(SEED)
_ref_paths = tr.path.to_numpy()[_rs.choice(len(tr), size=min(300, len(tr)), replace=False)]
_labs = [rgb2lab(np.asarray(Image.open(p).convert("RGB").resize((256, 256))) / 255.0).reshape(-1, 3) for p in _ref_paths]
_labs = np.concatenate(_labs); REF_MEAN, REF_STD = _labs.mean(0), _labs.std(0)
print("LC25000 LAB mean", REF_MEAN.round(2), "std", REF_STD.round(2))
def reinhard(img):
    lab = rgb2lab(np.asarray(img) / 255.0); m, s = lab.reshape(-1, 3).mean(0), lab.reshape(-1, 3).std(0) + 1e-6
    lab = (lab - m) / s * REF_STD + REF_MEAN
    return Image.fromarray((np.clip(lab2rgb(lab), 0, 1) * 255).astype(np.uint8))

def tiles(img):
    """LungHist700 images are 1200 x 1600 at 20x: cut into a 2 x 3 grid of near-square tiles.
    Chaoyang images are square: one tile. Each tile is resized to 224 x 224."""
    w, h = img.size
    if abs(w - h) / max(w, h) < 0.1: return [img]
    nr, nc = (2, 3) if w > h else (3, 2)
    return [img.crop((c * w // nc, r * h // nr, (c + 1) * w // nc, (r + 1) * h // nr)) for r in range(nr) for c in range(nc)]

class ExtDataset(Dataset):
    def __init__(self, frame, stain): self.p, self.y, self.stain = frame.path.tolist(), frame.y.tolist(), stain
    def __len__(self): return len(self.p)
    def __getitem__(self, i):
        img = Image.open(self.p[i]).convert("RGB")
        if self.stain: img = reinhard(img.resize((img.width // 2, img.height // 2)) if img.width > 1000 else img)
        return torch.stack([eval_tf(t) for t in tiles(img)]), self.y[i]

@torch.no_grad()
def ext_proba(m, frame, stain):
    m.eval(); out = []
    for i in range(len(frame)):
        x, _ = ExtDataset(frame.iloc[[i]], stain)[0]
        o = m(x.to(DEVICE)); o = o[0] if isinstance(o, tuple) else o
        out.append(F.softmax(o, 1).mean(0).cpu().numpy())        # average over tiles
    return np.stack(out)

LC25000 LAB mean [ 68.76  26.76 -30.62] std [16.51 16.19 21.25]


In [14]:
# ---------- external test: no training on these datasets ----------
from sklearn.metrics import balanced_accuracy_score, recall_score
res, PROBS = [], {}
for ds in ["LungHist700", "Chaoyang"]:
    fr = EXT[EXT.dataset == ds].reset_index(drop=True); y = fr.y.to_numpy(); allowed = EXT_CLASSES[ds]
    for mname, m in MODELS_EXT.items():
        for stain in [False, True]:
            t0 = time.time(); P = ext_proba(m, fr, stain); PROBS[(ds, mname, stain)] = P
            pr = np.array(allowed)[P[:, allowed].argmax(1)]          # restricted to the dataset's classes
            po = P.argmax(1)                                          # open: any of the 5 LC25000 classes
            rec = recall_score(y, pr, labels=allowed, average=None, zero_division=0)
            res.append(dict(dataset=ds, model=mname, stain_norm=stain, images=len(fr),
                            acc_restricted=round(accuracy_score(y, pr), 4),
                            bal_acc_restricted=round(balanced_accuracy_score(y, pr), 4),
                            macro_f1_restricted=round(f1_score(y, pr, labels=allowed, average="macro", zero_division=0), 4),
                            acc_open=round(accuracy_score(y, po), 4),
                            **{f"recall_{CLASSES[c]}": round(r, 4) for c, r in zip(allowed, rec)},
                            minutes=round((time.time() - t0) / 60, 1)))
            print(f"{ds:12s} {mname:22s} stain={str(stain):5s} acc {res[-1]['acc_restricted']:.4f}  "
                  f"bal_acc {res[-1]['bal_acc_restricted']:.4f}  open-acc {res[-1]['acc_open']:.4f}", flush=True)
            cm = confusion_matrix(y, pr, labels=allowed)
            pd.DataFrame(cm, index=[CLASSES[c] for c in allowed], columns=[CLASSES[c] for c in allowed]).to_csv(
                f"{OUT_DIR}/cm_{ds}_{'ours' if 'Mamba' in mname else 'resnet'}_{'stain' if stain else 'raw'}.csv")
EXT_RES = pd.DataFrame(res); EXT_RES.to_csv(f"{OUT_DIR}/table2_external_results.csv", index=False)
pd.set_option("display.width", 220, "display.max_columns", 30)
print("\n=== EXTERNAL VALIDATION (trained on LC25000 only, seed 42) ===\n" + EXT_RES.to_string(index=False))

LungHist700  CNN + BiMamba (ours)   stain=False acc 0.5237  bal_acc 0.5689  open-acc 0.4596
LungHist700  CNN + BiMamba (ours)   stain=True  acc 0.3900  bal_acc 0.4427  open-acc 0.1811
LungHist700  ResNet-50 (ImageNet)   stain=False acc 0.5070  bal_acc 0.5445  open-acc 0.0947
LungHist700  ResNet-50 (ImageNet)   stain=True  acc 0.5571  bal_acc 0.5813  open-acc 0.0195
Chaoyang     CNN + BiMamba (ours)   stain=False acc 0.6756  bal_acc 0.6449  open-acc 0.4973
Chaoyang     CNN + BiMamba (ours)   stain=True  acc 0.6229  bal_acc 0.6505  open-acc 0.3305
Chaoyang     ResNet-50 (ImageNet)   stain=False acc 0.7645  bal_acc 0.7378  open-acc 0.6406
Chaoyang     ResNet-50 (ImageNet)   stain=True  acc 0.8214  bal_acc 0.8013  open-acc 0.6209

=== EXTERNAL VALIDATION (trained on LC25000 only, seed 42) ===
    dataset                model  stain_norm  images  acc_restricted  bal_acc_restricted  macro_f1_restricted  acc_open  recall_lung_aca  recall_lung_scc  recall_lung_n  minutes  recall_colon_aca  rec

## XAI on external images

In [15]:
class InfluenceRollout:
    """Run 1's class-agnostic frozen-dynamics score. Kept as the 'before' baseline."""
    def __init__(self, model): self.model = model
    @staticmethod
    def _dir(cap, wbi=True):
        delta, A, B, u = cap["delta"], cap["A"], cap["B"], cap["u"]
        log_a = delta.unsqueeze(-1)*A[:,None,:]
        cum   = log_a.cumsum(2)
        decay = (2.0*(cum[:,:,-1:,:]-cum)).clamp(min=-60.0).exp()
        gain  = (delta.unsqueeze(-1)*B.transpose(1,2).unsqueeze(1))**2
        if wbi: gain = gain*(u.unsqueeze(-1)**2)
        return (decay*gain).sum(dim=(1,3))
    @staticmethod
    def _norm(s):
        s = s - s.amin(1, keepdim=True)
        return s/(s.amax(1, keepdim=True)+1e-8)
    def per_layer(self, x):
        self.model.eval()
        with torch.no_grad(): _, caps = self.model(x, capture=True)
        return [self._norm(self._dir(c["fwd"]) + self._dir(c["bwd"]).flip(1)) for c in caps]
    def __call__(self, x, residual=0.5):
        maps = self.per_layer(x)
        o = torch.ones_like(maps[0])
        for m in maps: o = o*(residual + (1-residual)*m)
        o = self._norm(o).view(-1,1,self.model.grid,self.model.grid)
        return F.interpolate(o, (IMG_SIZE,IMG_SIZE), mode="bilinear", align_corners=False)
    def spatial_spread(self, x):
        """Mean distance of the influence mass from its own centroid, per layer.
        Growing with depth = the Mamba stack really does widen its context."""
        maps = self.per_layer(x); g = self.model.grid
        yy, xx = torch.meshgrid(torch.arange(g).float(), torch.arange(g).float(), indexing="ij")
        coords = torch.stack([yy.flatten(), xx.flatten()], -1)
        out = []
        for m in maps:
            w = (m/(m.sum(1, keepdim=True)+1e-8)).cpu()
            d = (coords.unsqueeze(0) - (w @ coords).unsqueeze(1)).norm(dim=-1)
            out.append(float((w*d).sum(1).mean()))
        return out


class ClassConditionalInfluence:
    """Forward-pass, class-conditional frozen-dynamics relevance (Section 8)."""
    def __init__(self, model, use_prior=True, residual=0.5, mode="relu"):
        self.model, self.use_prior, self.residual, self.mode = model, use_prior, residual, mode
    @staticmethod
    def _reverse_state_sum(cap):
        """S_s = g_s*C_s + A_{s+1} * S_{s+1}, via one reversed associative scan."""
        a = (cap["delta"].unsqueeze(-1)*cap["A"][:,None,:]).exp()
        X = cap["gate"].unsqueeze(-1)*cap["C"].transpose(1,2).unsqueeze(1)
        return _inclusive_scan(_shift_right(a.flip(2),1,1.0), X.flip(2)).flip(2)
    def _dir(self, cap, a_cls):
        S = self._reverse_state_sum(cap)
        delta, B, u, D, gate = cap["delta"], cap["B"], cap["u"], cap["D"], cap["gate"]
        R = (torch.einsum("bdln,bnl->bdl", S, B)*delta*u*a_cls[:,:,None]).sum(1)
        return R + (D[None,:,None]*u*gate*a_cls[:,:,None]).sum(1)      # skip path
    def _class_vector(self, cap, target):
        """a_c = W_out^T (gamma * W_head[c]) -- linear, no autograd."""
        gamma = self.model.norm.weight.detach()
        return (self.model.head.weight.detach()[target]*gamma) @ cap["W_out"]
    @staticmethod
    def _norm(s):
        s = s - s.amin(1, keepdim=True)
        return s/(s.amax(1, keepdim=True)+1e-8)
    def __call__(self, x, target=None):
        self.model.eval()
        with torch.no_grad():
            logits, caps = self.model(x, capture=True)
            if target is None: target = logits.argmax(1)
            last = caps[-1]
            R = (self._dir(last["fwd"], self._class_vector(last["fwd"], target))
                 + self._dir(last["bwd"], self._class_vector(last["bwd"], target)).flip(1))
            R = self._norm(F.relu(R) if self.mode == "relu" else R.abs())
            if self.use_prior and len(caps) > 1:
                prior = torch.ones_like(R)
                for c in caps[:-1]:
                    m = InfluenceRollout._norm(InfluenceRollout._dir(c["fwd"])
                                               + InfluenceRollout._dir(c["bwd"]).flip(1))
                    prior = prior*(self.residual + (1-self.residual)*m)
                R = self._norm(R*prior)
        g = self.model.grid
        return F.interpolate(R.view(-1,1,g,g), (IMG_SIZE,IMG_SIZE),
                             mode="bilinear", align_corners=False)
print("explainers defined")

explainers defined


In [16]:
def grad_cam(model, x, target=None):
    model.eval()
    t = model.tokenize(x); t.requires_grad_(True)
    logits, _ = model.from_tokens(t, capture=False)
    if target is None: target = logits.argmax(1)
    grad = torch.autograd.grad(logits.gather(1, target[:,None]).sum(), t)[0]
    cam = F.relu((grad*t).sum(-1))
    cam = cam - cam.amin(1, keepdim=True); cam = cam/(cam.amax(1, keepdim=True)+1e-8)
    g = model.grid
    return F.interpolate(cam.view(-1,1,g,g), (IMG_SIZE,IMG_SIZE),
                         mode="bilinear", align_corners=False).detach()

def lime_map(model, x, target=None, n_samples=None, n_segments=50):
    from skimage.segmentation import slic
    from sklearn.linear_model import Ridge
    n_samples = n_samples or LIME_SAMPLES
    model.eval(); outs = []
    with torch.no_grad():
        bl = model(x)
        if isinstance(bl, tuple): bl = bl[0]
    tgt = bl.argmax(1) if target is None else target
    for i in range(x.shape[0]):
        img = x[i:i+1]; arr = img[0].permute(1,2,0).cpu().numpy()
        seg = slic(arr, n_segments=n_segments, compactness=10.0, channel_axis=-1, start_label=0)
        seg_t = torch.from_numpy(seg).to(x.device); k = int(seg.max())+1
        masks = np.random.binomial(1,0.5,(n_samples,k)).astype(np.float32); masks[0] = 1.0
        fill = img.mean(dim=(2,3), keepdim=True); probs = []
        with torch.no_grad():
            for b0 in range(0, n_samples, 16):
                batch = []
                for m in masks[b0:b0+16]:
                    keep = torch.from_numpy(m).to(x.device)[seg_t][None,None]
                    batch.append(img*keep + fill*(1-keep))
                lo = model(torch.cat(batch,0))
                if isinstance(lo, tuple): lo = lo[0]
                probs.append(F.softmax(lo,1)[:, tgt[i]].cpu().numpy())
        w = np.clip(Ridge(alpha=1.0).fit(masks, np.concatenate(probs)).coef_, 0, None)
        m = torch.from_numpy(w).float().to(x.device)[seg_t]
        m = m - m.min(); outs.append((m/(m.max()+1e-8))[None,None])
    return torch.cat(outs, 0)

def mamba_lrp(model, x, target=None):
    """MambaLRP relevance: gradient x input on the conservation-corrected forward pass."""
    model.eval()
    t = model.tokenize(x).detach().requires_grad_(True)
    logits = model.from_tokens_lrp(t)
    if target is None: target = logits.argmax(1)
    grad = torch.autograd.grad(logits.gather(1, target[:, None]).sum(), t)[0]
    R = F.relu((grad * t).sum(-1))                 # positive relevance, as Grad-CAM does
    R = R - R.amin(1, keepdim=True); R = R/(R.amax(1, keepdim=True)+1e-8)
    g = model.grid
    return F.interpolate(R.view(-1,1,g,g), (IMG_SIZE,IMG_SIZE),
                         mode="bilinear", align_corners=False).detach()


def plain_grad_input(model, x, target=None):
    """Gradient x input on the UNMODIFIED forward pass. Only used to prove that the
    MambaLRP detachments actually change something -- see the validation cell."""
    model.eval()
    t = model.tokenize(x).detach().requires_grad_(True)
    logits, _ = model.from_tokens(t, capture=False)
    if target is None: target = logits.argmax(1)
    grad = torch.autograd.grad(logits.gather(1, target[:, None]).sum(), t)[0]
    R = F.relu((grad * t).sum(-1)).detach()
    R = R - R.amin(1, keepdim=True); return R/(R.amax(1, keepdim=True)+1e-8)


# Every method now takes an EXPLICIT target class. This matters: under perturbation
# the prediction can flip, and comparing an explanation for class A against one for
# class B would measure the flip, not the method. The clean image's predicted class
# is therefore fixed and reused everywhere.
# influence_rollout ignores the target by construction -- it is class-agnostic, which
# is precisely the defect the class-conditional variants were built to fix.
METHODS = {
    "influence_rollout": lambda m,x,t: InfluenceRollout(m)(x),
    "influence_cc":      lambda m,x,t: ClassConditionalInfluence(m, use_prior=True)(x, t),
    "influence_cc_last": lambda m,x,t: ClassConditionalInfluence(m, use_prior=False)(x, t),
    "influence_cc_abs":  lambda m,x,t: ClassConditionalInfluence(m, mode="abs")(x, t),
    "grad_cam":          lambda m,x,t: grad_cam(m, x, t),
    "lime":              lambda m,x,t: lime_map(m, x, t),
    "mamba_lrp":         lambda m,x,t: mamba_lrp(m, x, t),
}
print("methods:", list(METHODS))

methods: ['influence_rollout', 'influence_cc', 'influence_cc_last', 'influence_cc_abs', 'grad_cam', 'lime', 'mamba_lrp']


In [17]:
import torchvision.transforms.functional as TF
from scipy.stats import spearmanr

PERTURB = {"low":dict(angle=5.0,noise=0.02,bright=0.00),
           "mid":dict(angle=10.0,noise=0.05,bright=0.10),
           "high":dict(angle=20.0,noise=0.10,bright=0.20)}

def perturb(x, cfg):
    o = TF.rotate(x, cfg["angle"])
    if cfg["bright"] > 0: o = o*(1.0+cfg["bright"])
    return o + torch.randn_like(o)*cfg["noise"]

def iou_topk(a, b, frac=TOP_FRAC):
    n = a.shape[0]; a, b = a.reshape(n,-1), b.reshape(n,-1)
    k = max(1, int(frac*a.shape[1]))
    ma = torch.zeros_like(a).scatter_(1, a.topk(k,1).indices, 1.0)
    mb = torch.zeros_like(b).scatter_(1, b.topk(k,1).indices, 1.0)
    return ((ma*mb).sum(1)/(((ma+mb)>0).float().sum(1)+1e-8)).cpu().numpy()

def stability(model, fn, x, level, target):
    """IoU between the clean map and the rotation-ALIGNED perturbed map.

    Both maps are computed for the SAME target class (the clean prediction). If the
    perturbed image were allowed to pick its own argmax, a prediction flip would make
    us compare explanations of two different classes -- measuring the flip, not the
    method's stability."""
    cfg = PERTURB[level]
    m0 = fn(model, x, target)
    m1 = TF.rotate(fn(model, perturb(x, cfg), target), -cfg["angle"])
    c = int(IMG_SIZE*0.15)                       # drop borders damaged by rotation
    return iou_topk(m0[...,c:-c,c:-c], m1[...,c:-c,c:-c])

@torch.no_grad()
def prediction_flip_rate(model, x, target, level):
    """How often perturbation changes the predicted class. Reported as a diagnostic:
    a high rate means the perturbation is not 'clinically irrelevant' after all."""
    lo = model(perturb(x, PERTURB[level]))
    if isinstance(lo, tuple): lo = lo[0]
    return (lo.argmax(1) != target).float().cpu().numpy()

@torch.no_grad()
def _curve(model, x, attr, target, steps, mode):
    n = x.shape[0]
    blur = F.conv2d(x, torch.ones(3,1,11,11,device=x.device)/121.0, padding=5, groups=3)
    order = attr.reshape(n,-1).argsort(1, descending=True); total = order.shape[1]
    ps = []
    for i in range(steps+1):
        k = int(total*i/steps)
        if mode == "deletion":
            m = torch.ones(n,total,device=x.device)
            if k: m.scatter_(1, order[:,:k], 0.0)
        else:
            m = torch.zeros(n,total,device=x.device)
            if k: m.scatter_(1, order[:,:k], 1.0)
        m = m.view(n,1,IMG_SIZE,IMG_SIZE)
        lo = model(x*m + blur*(1-m))
        if isinstance(lo, tuple): lo = lo[0]
        ps.append(F.softmax(lo,1).gather(1, target[:,None]).squeeze(1))
    return torch.stack(ps,1)

def sanity_check(model, fn, x, target, scope="full"):
    """Model-parameter randomisation (Adebayo et al.). LOW is good.

    scope="full"  -> CNN stem + patch embedding + every Mamba layer + head.
                     This is the honest 'whole network' test and the one to quote.
    scope="mamba" -> Mamba layers + head only; the CNN stem is left trained.
                     Reported alongside because the earlier version of this study
                     used that narrower scope, and the two are not comparable.

    The target class is held fixed at the clean model's prediction; a randomised
    model's own argmax is meaningless and would confound the comparison.
    """
    m0 = fn(model, x, target)
    rnd = copy.deepcopy(model)
    if scope == "full":
        mods = [rnd.stem, rnd.patch_embed] + list(rnd.layers) + [rnd.head]
    else:
        mods = list(rnd.layers) + [rnd.head]
    for mod in mods:
        for p in mod.parameters(): nn.init.normal_(p, std=0.05)
    m1 = fn(rnd, x, target); del rnd
    return iou_topk(m0, m1)

def true_gradient_map(model, x, target=None):
    model.eval()
    t = model.tokenize(x).detach().requires_grad_(True)
    logits, _ = model.from_tokens(t, capture=False)
    if target is None: target = logits.argmax(1)
    g = torch.autograd.grad(logits.gather(1, target[:,None]).sum(), t)[0]
    s = g.abs().sum(-1); s = s - s.amin(1, keepdim=True)
    return s/(s.amax(1, keepdim=True)+1e-8)

METRICS = ["deletion_auc","insertion_auc","iou_low","iou_mid","iou_high",
           "sanity_iou_full","sanity_iou_mamba","spearman_vs_gradient"]
HIGHER_IS_BETTER = {"deletion_auc":False, "insertion_auc":True, "iou_low":True,
                    "iou_mid":True, "iou_high":True, "sanity_iou_full":False,
                    "sanity_iou_mamba":False, "spearman_vs_gradient":True}
print("metrics:", METRICS)

metrics: ['deletion_auc', 'insertion_auc', 'iou_low', 'iou_mid', 'iou_high', 'sanity_iou_full', 'sanity_iou_mamba', 'spearman_vs_gradient']


In [18]:
# ---------- XAI on external images (same methods and metrics as the main notebook) ----------
from scipy.stats import wilcoxon
REPORT_METHODS = ["influence_cc", "grad_cam", "lime", "mamba_lrp"]
REPORT_METRICS = [("iou_low", True), ("iou_mid", True), ("iou_high", True), ("sanity_iou_full", False),
                  ("sanity_iou_mamba", False), ("insertion_auc", True), ("deletion_auc", False)]
xai_rows, xai_tests = [], []
for ds in ["LungHist700", "Chaoyang"]:
    fr = EXT[EXT.dataset == ds]; rs = np.random.RandomState(SEED)
    sel = pd.concat([g.iloc[rs.choice(len(g), size=min(XAI_PER_CLASS, len(g)), replace=False)] for _, g in fr.groupby("lc_class")])
    # one centre tile per image, stain-normalised (the setting with the better accuracy is reported below)
    Xs = []
    for p in sel.path:
        img = Image.open(p).convert("RGB"); t = tiles(img); t = t[len(t) // 2] if len(t) > 1 else t[0]
        Xs.append(eval_tf(reinhard(t)))
    X = torch.stack(Xs).to(DEVICE); allowed = EXT_CLASSES[ds]
    with torch.no_grad():
        lo = model(X); lo = lo[0] if isinstance(lo, tuple) else lo
    PRED = torch.as_tensor(np.array(allowed)[lo[:, allowed].argmax(1).cpu().numpy()], device=DEVICE)
    per = {}
    for name in REPORT_METHODS:
        set_seed(SEED); fn = METHODS[name]; acc_ = defaultdict(list); t0 = time.time()
        for i in range(0, X.shape[0], 4):
            xb, pb = X[i:i+4], PRED[i:i+4]; attr = fn(model, xb, pb)
            dc = _curve(model, xb, attr, pb, DELETION_STEPS, "deletion").cpu().numpy()
            ic = _curve(model, xb, attr, pb, DELETION_STEPS, "insertion").cpu().numpy()
            acc_["deletion_auc"].append(dc.mean(1)); acc_["insertion_auc"].append(ic.mean(1))
            for lv in PERTURB: acc_[f"iou_{lv}"].append(stability(model, fn, xb, lv, pb))
            acc_["sanity_iou_full"].append(sanity_check(model, fn, xb, pb, "full"))
            acc_["sanity_iou_mamba"].append(sanity_check(model, fn, xb, pb, "mamba"))
        per[name] = {k: np.concatenate(v) for k, v in acc_.items()}
        xai_rows.append(dict(dataset=ds, method=name, images=X.shape[0], minutes=round((time.time()-t0)/60, 1),
                             **{k: round(float(v.mean()), 4) for k, v in per[name].items()}))
        print(f"{ds:12s} {name:14s}", {k: round(float(v.mean()), 3) for k, v in per[name].items()}, flush=True)
    for metric, hib in REPORT_METRICS:
        ps, res_ = [], []
        for o in ["grad_cam", "lime", "mamba_lrp"]:
            a, b = per[REFERENCE][metric], per[o][metric]
            try: p = wilcoxon(a, b, zero_method="zsplit").pvalue
            except ValueError: p = 1.0
            ps.append(p); res_.append((o, a.mean(), b.mean()))
        order = np.argsort(ps); adj = np.empty(3); run = 0.0
        for r_, i_ in enumerate(order): run = max(run, (3 - r_) * ps[i_]); adj[i_] = min(run, 1.0)
        for (o, am, bm), pa in zip(res_, adj):
            better = (am > bm) if hib else (am < bm)
            xai_tests.append(dict(dataset=ds, metric=metric, other=o, ours=round(am, 4), other_mean=round(bm, 4),
                                  p_holm=float(f"{pa:.3g}"), result=("win" if better else "loss") if pa < ALPHA else "tie"))
XAI_TAB = pd.DataFrame(xai_rows); XAI_TAB.to_csv(f"{OUT_DIR}/table3_external_xai.csv", index=False)
XAI_TEST = pd.DataFrame(xai_tests); XAI_TEST.to_csv(f"{OUT_DIR}/table4_external_xai_tests.csv", index=False)
print("\n" + XAI_TEST.to_string(index=False))
for ds in ["LungHist700", "Chaoyang"]:
    v = XAI_TEST[XAI_TEST.dataset == ds].result.value_counts()
    print(f"{ds}: influence_cc  {v.get('win', 0)} wins, {v.get('tie', 0)} ties, {v.get('loss', 0)} losses")

LungHist700  influence_cc   {'deletion_auc': 0.235, 'insertion_auc': 0.236, 'iou_low': 0.577, 'iou_mid': 0.463, 'iou_high': 0.359, 'sanity_iou_full': 0.119, 'sanity_iou_mamba': 0.153}
LungHist700  grad_cam       {'deletion_auc': 0.264, 'insertion_auc': 0.187, 'iou_low': 0.41, 'iou_mid': 0.323, 'iou_high': 0.274, 'sanity_iou_full': 0.114, 'sanity_iou_mamba': 0.139}
LungHist700  lime           {'deletion_auc': 0.196, 'insertion_auc': 0.259, 'iou_low': 0.311, 'iou_mid': 0.209, 'iou_high': 0.28, 'sanity_iou_full': 0.227, 'sanity_iou_mamba': 0.382}
LungHist700  mamba_lrp      {'deletion_auc': 0.255, 'insertion_auc': 0.193, 'iou_low': 0.473, 'iou_mid': 0.392, 'iou_high': 0.325, 'sanity_iou_full': 0.12, 'sanity_iou_mamba': 0.145}
Chaoyang     influence_cc   {'deletion_auc': 0.287, 'insertion_auc': 0.351, 'iou_low': 0.569, 'iou_mid': 0.423, 'iou_high': 0.316, 'sanity_iou_full': 0.126, 'sanity_iou_mamba': 0.11}
Chaoyang     grad_cam       {'deletion_auc': 0.292, 'insertion_auc': 0.316, 'iou_low

In [19]:
# ---------- figure + summary ----------
fig, axs = plt.subplots(1, 2, figsize=(12, 4.2))
for ax, ds in zip(axs, ["LungHist700", "Chaoyang"]):
    sub = EXT_RES[EXT_RES.dataset == ds]; labels = []
    for i, (_, r) in enumerate(sub.iterrows()):
        ax.bar(i, r.acc_restricted * 100, color="#2a78d6" if "Mamba" in r.model else "#eb6834")
        ax.text(i, r.acc_restricted * 100 + 1, f"{r.acc_restricted*100:.1f}", ha="center", fontsize=9)
        labels.append(f"{'Ours' if 'Mamba' in r.model else 'ResNet-50'}\n{'stain-norm' if r.stain_norm else 'raw'}")
    ax.set_xticks(range(len(labels)), labels, fontsize=8); ax.set_ylim(0, 105); ax.set_ylabel("accuracy (%)")
    ax.set_title(f"{ds}: external test (trained on LC25000 only)"); ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/fig_external_accuracy.png", dpi=200); plt.close()
print("weights used:", WEIGHT_SOURCE)
print("saved:", sorted(os.listdir(OUT_DIR)))

weights used: trained in this notebook
saved: ['cm_Chaoyang_ours_raw.csv', 'cm_Chaoyang_ours_stain.csv', 'cm_Chaoyang_resnet_raw.csv', 'cm_Chaoyang_resnet_stain.csv', 'cm_LungHist700_ours_raw.csv', 'cm_LungHist700_ours_stain.csv', 'cm_LungHist700_resnet_raw.csv', 'cm_LungHist700_resnet_stain.csv', 'fig_external_accuracy.png', 'lc25000_image_groups.csv', 'table0_lc25000_test.csv', 'table1_external_data.csv', 'table2_external_results.csv', 'table3_external_xai.csv', 'table4_external_xai_tests.csv', 'weights_cnn_mamba_refgroup.pt', 'weights_resnet50_lc25000_seed42.pt']


## How to report

* Write: *"External validation: the CNN + BiMamba trained on LC25000 (seed 42) was tested without any
  re-training on LungHist700 (3 lung classes) and on the normal and adenocarcinoma classes of Chaoyang."*
* Main number = `acc_restricted` (the model chooses among the classes that exist in that dataset).
  Report raw and stain-normalised results; say which one is higher.
* Always show ResNet-50 next to ours: a drop for **both** models means a dataset shift
  (other hospital, staining, scanner, magnification), not a weakness of one model.
* `table4_external_xai_tests.csv` gives win / tie / loss of influence_cc on external images.